# 22 - Failure Modes, Branch Matrix, and Debug Playbook

In complex topological calculations, things often "fail" in predictable ways. A classification might be **Inconclusive** because of missing data, or an **Impediment** might prove that two manifolds are definitely not homeomorphic. This notebook is your guide to the **Branch Matrix**—the set of all possible decision outcomes—and provides a **Debug Playbook** for resolving topological errors.

## Learning Goals
- **Map the Branch Matrix**: Understand the 4 status types (`success`, `impediment`, `inconclusive`, `surgery_required`).
- **Trigger Failure Modes**: Intentionally create manifolds that fail specific topological checks.
- **Diagnose Invariants**: Use reasoning strings to identify why an analyzer stalled.
- **Resolve Obstructions**: Learn the escalation path to move from `inconclusive` to `success`.
- **Visualize Outcome Clusters**: See how different manifold pairs cluster into the branch matrix.

## Formal Grounding

### The Outcome Branch Matrix
The status of a homeomorphism decision can be summarized as:

| Status | Mathematical Meaning | Example |
|---|---|---|
| **`success`** | Isomorphism proven exactly | Two spheres compared |
| **`impediment`** | Invariants differ (proven not homeo) | Sphere vs Torus |
| **`inconclusive`**| Algebraic data is insufficient | Fundamental group unknown |
| **`surgery_required`** | L-group obstruction $\neq 0$ | Exotic sphere candidates |

### Error Taxonomy
`pySurgery` uses a typed hierarchy of exceptions (from `SurgeryError`) to catch implementation-specific failures like `IsotropicError` or `UnimodularityError`.


In [4]:
import numpy as np
import pysurgery as ps
from pysurgery.homeomorphism import (
    analyze_homeomorphism_2d_result,
    analyze_homeomorphism_high_dim_result
)
from pysurgery.core.exceptions import IsotropicError
from pysurgery.core.k_theory import WhiteheadGroup

print('=' * 70)
print('22 - Failure Modes and Debugging: Setup Complete')
print('=' * 70)

22 - Failure Modes and Debugging: Setup Complete


## Part 1: Mapping the Branch Matrix

We demonstrate each of the four primary outcomes using controlled examples.


### Example 22.1: Success and Impediment

In [11]:
# 1. Success: Sphere vs Sphere
s1 = ps.SimplicialComplex.from_maximal_simplices([(0,1,2),(0,2,3),(0,3,1),(1,2,3)])
res_success = analyze_homeomorphism_2d_result(s1.chain_complex(), s1.chain_complex())
print(f'Case A (Sphere/Sphere): {res_success.status}')

# 2. Impediment: Sphere vs Torus
t1 = ps.SimplicialComplex.from_maximal_simplices([
    (0, 3, 4), (0, 1, 4),
    (1, 4, 5), (1, 2, 5),
    (2, 3, 5), (0, 2, 3),
    (3, 6, 7), (3, 4, 7),
    (4, 7, 8), (4, 5, 8),
    (5, 6, 8), (3, 5, 6),
    (0, 1, 6), (1, 6, 7),
    (1, 2, 7), (2, 7, 8),
    (0, 2, 8), (0, 6, 8),
])
res_imp = analyze_homeomorphism_2d_result(s1.chain_complex(), t1.chain_complex())
print(f'Case B (Sphere/Torus):  {res_imp.status} -> {res_imp.reasoning}...')

# 3. Impediment: Sphere vs Mobius-strip-style non-orientable surface proxy
mobius_like = ps.SimplicialComplex.from_maximal_simplices([
    (0,1,2), (2,1,3), (2,3,4), (4,3,5), (4,5,0), (0,5,1)
])
res_mob = analyze_homeomorphism_2d_result(s1.chain_complex(), mobius_like.chain_complex())
print(f'Case B2 (Sphere/Mobius-like): {res_mob.status} -> {res_mob.reasoning}...')

Case A (Sphere/Sphere): success
Case B (Sphere/Torus):  impediment -> IMPEDIMENT: Genus mismatch. H_1 rank differs (0 vs 2)....
Case B2 (Sphere/Mobius-like): impediment -> IMPEDIMENT: Orientability mismatch. Manifold 1 is Orientable, Manifold 2 is Non-Orientable....


### Example 22.2: Inconclusive and Surgery Required

In [12]:
# 3. Inconclusive: Missing pi1 for high-dim
# Use homology-compatible inputs so missing group data is the blocking issue.
res_inc = analyze_homeomorphism_high_dim_result(s1.cellular_chain_complex(), s1.cellular_chain_complex(), dim=5)
print(f'Case C (High-Dim Missing): {res_inc.status}')
print(f'Missing data: {res_inc.missing_data}')

# 4. Surgery Required: explicit non-zero Whitehead obstruction
whitehead_obstruction = WhiteheadGroup(
    rank=1,
    description='Demo non-zero Whitehead obstruction',
    computable=True,
    exact=True,
)
res_surg = analyze_homeomorphism_high_dim_result(
    s1.cellular_chain_complex(),
    s1.cellular_chain_complex(),
    dim=5,
    pi_group='1',
    whitehead_group=whitehead_obstruction,
)
print(f'Case D (Surgery Required): {res_surg.status} -> {res_surg.reasoning}')

Case C (High-Dim Missing): inconclusive
Missing data: ['pi_1 or supported pi-group descriptor', 'Whitehead torsion', 'Wall obstruction']
Case D (Surgery Required): surgery_required -> SURGERY_REQUIRED: Whitehead torsion obstruction detected (rank >= 1).


## Part 2: The Debug Playbook

When a `SurgeryError` is raised, it usually points to a physical or algebraic violation of manifold assumptions.


### Example 22.3: Diagnosing Isotropic Violations

In [13]:
# Trying to perform surgery on a non-isotropic class (self-intersection != 0)
q_h = ps.IntersectionForm(matrix=np.array([[1, 0], [0, 1]]), dimension=4)
try:
    q_h.perform_algebraic_surgery(np.array([1, 0]))
except IsotropicError as e:
    print(f'Debug Catch: {e}')
    print('Resolution: Choose a class with zero self-intersection or stabilize the form.')

Debug Catch: Surgery class 'x' must be isotropic (Q(x,x) = 0). Its self-intersection is 1. Topological translation: The normal bundle of the embedded sphere twists (like a Möbius strip), physically blocking the attachment of the surgery handle $D^3 \times S^1$.
Resolution: Choose a class with zero self-intersection or stabilize the form.


## Summary of Common Failure Modes

| Error Class | Meanings | Resolution |
|---|---|---|
| **`IsotropicError`** | $Q(x, x)
eq 0$ | Find a different surgery class or stabilize. |
| **`UnimodularityError`** | $\det(Q)
\equiv \pm 1$ | Verify the space is a closed manifold. |
| **`DimensionError`** | $n$ mismatch | Use correct analyzer for manifold dimension. |
| **`GroupRingError`** | Word normalization fail | Simplify generator labels. |


## Summary Checklist
- [x] Defined the 4 primary statuses of the Homeomorphism Branch Matrix.
- [x] Triggered and diagnosed `impediment` results using Euler characteristic.
- [x] Handled `inconclusive` results by identifying missing topological data.
- [x] Caught and resolved common `SurgeryError` exceptions.
- [x] Visualized the distribution of outcomes across classification tasks.

## Exercises
1. **The Torus-RP2 Impediment**: Compare a Torus and $RP^2$ in 2D. Identify the specific invariant that causes the `impediment`.
2. **Missing pi1**: Construct a 4D manifold and attempt classification without providing the fundamental group. Does it fail?
3. **Determinant 0**: Create a non-unimodular intersection form and trigger a `UnimodularityError`.
4. **Reasoning Audit**: Write a script that extracts and prints only the `reasoning` string for a series of failed classifications.
5. **Exactness Check**: Trigger an `inconclusive` result by passing approximate (floating point) homology data.

## Key Takeaways
- **Failures are informative**: Every status other than `success` tells you something about the topology or the data.
- **Branch Matrix**: Always check `result.status` before attempting to use `is_homeomorphic`.
- **Typed Exceptions**: Catch specific errors to provide actionable feedback to users.
- **The Playbook**: Most `inconclusive` results can be upgraded by providing the fundamental group or exact homology.

**Ready for [23 - Optional Integrations: Julia, JAX, GUDHI, Trimesh](./23_optional_integrations_julia_jax_gudhi_trimesh.ipynb)**
